In [6]:
import ollama
import os
from pathlib import Path
import re
from IPython.display import Markdown, display

In [7]:
# 链接 ollama
client = ollama.Client(host='http://127.0.0.1:11435')

In [8]:
# ===== 参数 =====
QUESTIONS_DIR = Path("Questions")
# MODEL_NAME = "llama4:scout"
MODEL_NAME = "qwen3-coder-next"

prompt_path = Path("prompt2.txt")
system_prompt = prompt_path.read_text(encoding="utf-8")


def list_question_dirs(questions_dir: Path):
    return sorted(
        [p for p in questions_dir.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda x: x.name
    )


def list_explain_txts(explain_dir: Path):
    txts = []

    for p in explain_dir.iterdir():
        if not (p.is_file() and p.suffix.lower() == ".txt"):
            continue

        parts = p.stem.split()

        # 文件名格式："{global_idx} {temperature}.txt"
        if len(parts) == 2 and parts[0].isdigit():
            txts.append((int(parts[0]), p))

    txts_sorted = sorted(txts, key=lambda x: x[0])
    return [p for _, p in txts_sorted]


def parse_idx_and_temp(txt_path: Path):
    parts = txt_path.stem.split()
    if len(parts) != 2 or not parts[0].isdigit():
        raise ValueError(f"非法文件名格式: {txt_path.name}")

    global_idx = int(parts[0])
    temperature = float(parts[1])
    return global_idx, temperature

def extract_python_code_from_llm_output(text: str) -> str:
    """
    从 LLM 输出中提取可直接保存为 .py 的代码。

    处理规则：
    1. 如果存在 ```python ... ``` 或 ``` ... ``` 代码块，优先提取第一个代码块内容
    2. 否则返回原文本
    3. 去掉首尾空白
    """
    text = text.strip()

    # 优先匹配 ```python ... ```
    m = re.search(r"```python\s*\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # 再匹配普通 ``` ... ```
    m = re.search(r"```\s*\n(.*?)```", text, flags=re.DOTALL)
    if m:
        return m.group(1).strip()

    return text

dirs = list_question_dirs(QUESTIONS_DIR)

In [9]:
display(dirs)

[PosixPath('Questions/Easy B3666'),
 PosixPath('Questions/Easy P15288'),
 PosixPath('Questions/Easy P15457'),
 PosixPath('Questions/Easy P4306'),
 PosixPath('Questions/Easy P7714'),
 PosixPath('Questions/Hard P11658'),
 PosixPath('Questions/Hard P11823'),
 PosixPath('Questions/Hard P13901'),
 PosixPath('Questions/Hard P15082'),
 PosixPath('Questions/Hard P6845'),
 PosixPath('Questions/ML Q1'),
 PosixPath('Questions/ML Q2'),
 PosixPath('Questions/ML Q3'),
 PosixPath('Questions/Mid P1407'),
 PosixPath('Questions/Mid P14989'),
 PosixPath('Questions/Mid P3007'),
 PosixPath('Questions/Mid P3167'),
 PosixPath('Questions/Mid P4092')]

In [ ]:
# 对所有题目遍历输出code
for d in dirs:
    explain_dir = d / "LLM Explains"
    code_dir = d / "LLM Codes"
    code_dir.mkdir(parents=True, exist_ok=True)

    file_num = 0
    token = 0

    if not explain_dir.exists():
        print(f"skip: {explain_dir} not found")
        continue

    explain_files = list_explain_txts(explain_dir)

    for explain_path in explain_files:
        try:
            global_idx, temperature = parse_idx_and_temp(explain_path)
            user_input = explain_path.read_text(encoding="utf-8")

            response = ollama.chat(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input},
                ],
                options={
                    "temperature": temperature,
                },
            )

            raw_output_text = response["message"]["content"]
            output_text = extract_python_code_from_llm_output(raw_output_text)
            
            output_path = code_dir / f"{global_idx} {temperature}.py"
            output_path.write_text(output_text, encoding="utf-8")

            print(f"done: {d.name} -> {output_path.name}")
            file_num += 1
            token += response["prompt_eval_count"] + response["eval_count"]

        except Exception as e:
            print(f"error: {d.name}, file={explain_path.name} -> {e}")
    print(f"总token消耗：{token}，平均token消耗：{token/file_num}")

In [ ]:
# 测试index0全部解释，每个输出一个code
d = dirs[4]

explain_dir = d / "LLM Explains"
code_dir = d / "LLM Codes"
code_dir.mkdir(parents=True, exist_ok=True)

file_num = 0
token = 0

if not explain_dir.exists():
    print(f"skip: {explain_dir} not found")
else:
    explain_files = list_explain_txts(explain_dir)

    for explain_path in explain_files:
        try:
            global_idx, temperature = parse_idx_and_temp(explain_path)
            user_input = explain_path.read_text(encoding="utf-8")

            response = ollama.chat(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input},    # 只喂解释
                    # {"role": "user", "content": (d / "Question.txt").read_text(encoding="utf-8")},    # 只喂原题
                    # {"role": "user", "content": (d / "Question.txt").read_text(encoding="utf-8") + user_input},    # 原题+解释
                ],
                options={
                    "temperature": temperature,
                },
            )

            raw_output_text = response["message"]["content"]
            output_text = extract_python_code_from_llm_output(raw_output_text)
            
            output_path = code_dir / f"{global_idx} {temperature}.py"
            output_path.write_text(output_text, encoding="utf-8")

            print(f"done: {d.name} -> {output_path.name}")
            file_num += 1
            token += response["prompt_eval_count"] + response["eval_count"]

        except Exception as e:
            print(f"error: {d.name}, file={explain_path.name} -> {e}")

print(f"总token消耗：{token}，平均token消耗：{token/file_num}")